In [2]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split

SEED = 42
RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")

SIGNALS = ["signal1", "signal2", "signal3"]
COLORS = ["#0072B2", "#E69F00", "#009E73"]  # Okabe-Ito, colorblind-safe

print(f"scikit-learn {sklearn.__version__}")  # the split is reproducible for a given version

scikit-learn 1.9.1


In [5]:
df = pd.read_csv(RAW_DIR/"Round1_Diagnostic_33.csv")

# Assuming your dataframe is named `df`
# Columns present: id, own, competing1, competing2, borrowertype, accepted, default, profit

# 1. Determine acceptance for competitors (accepted1, accepted2)
# Assuming the borrower chooses the lowest interest rate among the three lenders:
rates = df[['own', 'competing1', 'competing2']]
min_rate_col = rates.idxmin(axis=1)

# If 'accepted' is already given for us, we can check if it aligns or use the lowest-rate rule for all:
df['accepted1'] = (min_rate_col == 'competing1').astype(int)
df['accepted2'] = (min_rate_col == 'competing2').astype(int)

# If competitors also have a credit acceptance filter similar to yours, 
# you can mask acceptance where default/risk thresholds apply (uncomment if needed):
# df['accepted1'] = df['accepted1'] & (df['competing1_valid_condition'])
# df['accepted2'] = df['accepted2'] & (df['competing2_valid_condition'])


# 2. Estimate or calculate profit1 and profit2
# Since you have your own 'profit' column, we can deduce the profit per unit rate or use your profit structure.
# Assuming profit scales linearly with the proposed rate (or you can substitute your exact profit formula):
# Let's extract the profit per unit of rate from your rows where accepted == 1:
# profit = accepted * (rate * factor_if_no_default - loss_if_default)

# A clean way to approximate/calculate competitor profit using your existing profit logic:
# If you know the multiplier (e.g. loan amount * margin), apply it to competing1 and competing2:
# For example, if profit = accepted * (rate * multiplier - default_penalty):
# Let's derive the multiplier from your data if possible, or use a standard proportional formula:

# Let's assume profit formula structure: 
# If accepted == 1: profit = rate * positive_multiplier (if default == 0) else -negative_penalty
# To keep it exact based on your existing `profit` column's scale:
# We can compute the implied multiplier where accepted == 1:
mask_us = df['accepted'] == 1
# Average revenue per unit rate when accepted:
# (You can adjust this based on your exact simulation parameters)

# General approach to compute profit1 and profit2 matching your profit's scale:
# If profit per unit rate is consistent, we can estimate it:
unit_profit = df.loc[mask_us & (df['default'] == 0), 'profit'] / df.loc[mask_us & (df['default'] == 0), 'own']
avg_unit_profit = unit_profit.mean() if not unit_profit.empty else 1.0

default_loss = df.loc[mask_us & (df['default'] == 1), 'profit'].abs()
avg_default_loss = default_loss.mean() if not default_loss.empty else 100.0

# Calculate profit for Competitor 1
df['profit1'] = np.where(
    df['accepted1'] == 1,
    np.where(df['default'] == 0, df['competing1'] * avg_unit_profit, -avg_default_loss),
    0
)

# Calculate profit for Competitor 2
df['profit2'] = np.where(
    df['accepted2'] == 1,
    np.where(df['default'] == 0, df['competing2'] * avg_unit_profit, -avg_default_loss),
    0
)

# 3. Sum the profits to compare and identify teams
total_our_profit = df['profit'].sum()
total_profit1 = df['profit1'].sum()
total_profit2 = df['profit2'].sum()

print(f"Our Total P&L: {total_our_profit:,.2f}M€")
print(f"Competitor 1 Total P&L: {total_profit1:,.2f}M€")
print(f"Competitor 2 Total P&L: {total_profit2:,.2f}M€")

FileNotFoundError: [Errno 2] No such file or directory: '..\\data\\raw\\Round1_Diagnostic_33.csv'